In [ ]:
from transformers import ViTForImageClassification, ViTFeatureExtractor, Trainer, TrainingArguments
from PIL import Image
import torch
import pandas as pd
import os
from torch.utils.data import Dataset, random_split

In [ ]:
#google/vit-huge-patch14-224-in21k 0.9888
#google/vit-base-patch32-384 0.986

In [ ]:
model_name = "google/vit-base-patch32-384"
model = ViTForImageClassification.from_pretrained(model_name, num_labels=2,ignore_mismatched_sizes=True)
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)

class HouseDataset(Dataset):
    def __init__(self, df, image_folder, feature_extractor):
        self.df = df
        self.image_folder = image_folder
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_folder, self.df.iloc[idx]["image_name"])
        image = Image.open(image_path).convert("RGB")
        inputs = self.feature_extractor(images=image, return_tensors="pt")
        label = torch.tensor(self.df.iloc[idx]["class"], dtype=torch.long)
        return {"pixel_values": inputs["pixel_values"].squeeze(), "labels": label}

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch32-384 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [ ]:
df = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv")
image_folder = "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train"
dataset = HouseDataset(df, image_folder, feature_extractor)

In [ ]:
train_size = int(0.8 * len(dataset))
eval_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,  # Log progress every 10 steps
    report_to="none",
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics  # 👈 เพิ่มตรงนี้
)


In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.111700,0.179369,0.930626
2,0.068200,0.170074,0.940778
3,0.009700,0.185130,0.940778
4,0.002500,0.183753,0.939086


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=1184, training_loss=0.09955587032537379, metrics={'train_runtime': 539.5944, 'train_samples_per_second': 17.509, 'train_steps_per_second': 2.194, 'total_flos': 2.1949792915370803e+18, 'train_loss': 0.09955587032537379, 'epoch': 4.0})

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def predict(images_folder, model, feature_extractor, device):
    model.eval()
    predictions = []
    image_files = [f for f in os.listdir(images_folder) if f.endswith(".jpg") or f.endswith(".png")]

    for image_file in image_files:
        image_path = os.path.join(images_folder, image_file)
        image = Image.open(image_path).convert("RGB")
        inputs = feature_extractor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_label = torch.argmax(outputs.logits, dim=-1).item()
        predictions.append((image_file, predicted_label))

    return predictions

In [ ]:
test_folder = "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test"
predictions = predict(test_folder, model, feature_extractor,device)

In [ ]:
predictions

[('fd0900c0.jpg', 1),
 ('b17ced93.jpg', 0),
 ('5157729e.jpg', 1),
 ('9b25615e.jpg', 0),
 ('1df1da60.jpg', 0),
 ('43c5b7da.jpg', 1),
 ('7e90f408.jpg', 1),
 ('5f5ef892.jpg', 0),
 ('7e6cafaa.jpg', 1),
 ('4da96bd3.jpg', 1),
 ('4bc363dd.jpg', 0),
 ('d1b21893.jpg', 1),
 ('711bde86.jpg', 0),
 ('a5261896.jpg', 0),
 ('12c17ddf.jpg', 1),
 ('d18836ca.jpg', 1),
 ('57b3c5c2.jpg', 0),
 ('2d7181b5.jpg', 0),
 ('57200181.jpg', 1),
 ('41487df6.jpg', 1),
 ('c3726bb5.jpg', 0),
 ('c9d1a911.jpg', 0),
 ('5f033aaf.jpg', 1),
 ('4961b93c.jpg', 0),
 ('d1d12715.jpg', 1),
 ('beefc26e.jpg', 1),
 ('01ccbb76.jpg', 0),
 ('c37dbcf4.jpg', 0),
 ('35d7632c.jpg', 0),
 ('228369aa.jpg', 1),
 ('dfcba56a.jpg', 1),
 ('36cf3714.jpg', 0),
 ('7a920b80.jpg', 0),
 ('9bfc35a3.jpg', 1),
 ('f31a3abc.jpg', 1),
 ('4fd38ae5.jpg', 1),
 ('d8d67dcf.jpg', 1),
 ('0f198b7d.jpg', 0),
 ('c74652c2.jpg', 0),
 ('887f5c9f.jpg', 0),
 ('a799221f.jpg', 1),
 ('c336ce30.jpg', 1),
 ('39b8b3bb.jpg', 1),
 ('504c8cfe.jpg', 0),
 ('defe8dfd.jpg', 0),
 ('b6f685c

In [ ]:
submit = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/sample_submission.csv")

In [ ]:
submit

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,id,answer
0,e4b420b0,0.0
1,23efa479,0.0
2,1f0f2402,0.0
3,8a60480c,NaN
4,11f20127,NaN
...,...,...
1545,ed44ac4d,NaN
1546,a2067109,NaN
1547,1481178a,NaN
1548,340a3e9d,NaN


In [ ]:
ss = pd.DataFrame(predictions)

In [ ]:
ss = ss.rename(columns={0: "id", 1: "answer"})
ss["id"] = ss["id"].str.replace(".jpg", "")

In [ ]:
submit = submit.set_index("id")
ss = ss.set_index("id")
df_sorted = ss.reindex(submit.index)

In [ ]:
df_sorted

,answer
id,
e4b420b0,0
23efa479,0
1f0f2402,0
8a60480c,0
11f20127,0
...,...
ed44ac4d,1
a2067109,1
1481178a,1


In [ ]:
df_sorted.to_csv("submit_house.csv")